# Bayesian Optimization Workshop

## Hands-on Guide to Designing Experiments with BO

Welcome! This notebook will guide you through the Bayesian optimization workflow step by step.

### What you'll learn:
1. How to generate an initial experimental design
2. How to train a machine learning model on your results
3. How to get AI-suggested next experiments

### How to use this notebook:
- **Edit only the cells marked with `✏️ EDIT THIS`**
- Run cells in order (Shift + Enter)
- Don't worry about the code - just follow the instructions!

---

## Step 1: Setup

Run this cell to import all necessary libraries. You don't need to modify anything here.

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime

# Import project modules
from acquisition.utils import generate_initial_design
from data.preprocessing import prepare_data
from models import GPModel, fit_gp_model
from botorch.models import ModelListGP
from botorch.sampling import SobolQMCNormalSampler
from botorch.utils.transforms import unnormalize
from acquisition import create_qnehvi_acquisition, optimize_qnehvi, get_urea_constraint_callable
from config import ConstraintConfig, get_normalized_bounds
import matplotlib.pyplot as plt

# Import synthetic functions for demo mode
from demo_workflow import synthetic_objective_1, synthetic_objective_2

print("✅ All libraries loaded successfully!")
print("\nLet's start designing your experiments!")

---
## 🔄 Resume Session

**Closed the notebook and coming back?** Here's how to continue:

1. **Run Step 1** (imports)
2. **Run Step 2** (configuration) - make sure `PARAMETERS` and `CAMPAIGN_NAME` match your previous session
3. **Skip to the step you need:**
   - **Step 4**: Load your experimental results
   - **Step 5**: Train the model
   - **Step 6**: Get new suggestions
   - **Step 7**: Combine data

The notebook **auto-detects** your progress from existing files - no need to track iteration numbers!

**Real experiment workflow:**
```
Day 1: Steps 1-3 → Get initial design → Close notebook
Day 2: Steps 1-2 → Step 4 → Enter results → Steps 5-6 → Get suggestions → Close notebook
Day 3: Steps 1-2 → Step 7 → Combine data → Steps 5-6 → Get suggestions → Close notebook
...repeat...
```

---

---
## Step 2: Define Your Experiment

### ✏️ EDIT THIS SECTION

Fill in your experiment parameters below. This defines what you want to optimize.

**Example:**
```python
"Temperature": (10, 25),  # Temperature in °C, from 10 to 25
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ✏️ EDIT THIS: Define your experimental parameters
# ═══════════════════════════════════════════════════════════════════════════

# Parameter names and their bounds (minimum, maximum)
# Format: "Parameter Name": (min_value, max_value)
PARAMETERS = {
    "DTT [mM]": (0, 25),           # DTT concentration in mM
    "GSSG [mM]": (0, 2.5),         # GSSG concentration in mM
    "Dilution Factor": (2, 40),    # Dilution factor
    "pH": (8, 11),                 # pH value
    "Final Urea [M]": (0, 6),      # Urea concentration in M
}

# Names of the objectives (what you want to measure)
# These will be the column names for your results
OBJECTIVE_NAMES = [
    "Delta AEW",    # First objective (e.g., yield)
    "p_proxy",      # Second objective (e.g., purity)
]

# ═══════════════════════════════════════════════════════════════════════════
# ✏️ EDIT THIS: Optimization settings
# ═══════════════════════════════════════════════════════════════════════════

# How many experiments to generate initially?
N_INITIAL_SAMPLES = 12

# How many new experiments should BO suggest each round?
N_CANDIDATES = 4

# Noise level for the model (higher = more uncertainty in measurements)
# Typical values: 0.01 (very precise) to 0.15 (quite noisy measurements)
NOISE_LEVEL = 0.05

# Number of training iterations (more = better fit, but slower)
N_TRAINING_ITERATIONS = 1000

# Random seed for reproducibility
RANDOM_SEED = 42

# Turn this off in case the urea / dilution constraint is not relevant for your optimization problem
ConstraintConfig.ENABLE_UREA_CONSTRAINT = True

# ═══════════════════════════════════════════════════════════════════════════
# ✏️ EDIT THIS: Where to save your results
# ═══════════════════════════════════════════════════════════════════════════

# Folder name for your campaign (will be created automatically)
CAMPAIGN_NAME = "my_first_campaign"

# ═══════════════════════════════════════════════════════════════════════════
# ✏️ EDIT THIS: Demo mode (for practice without real experiments)
# ═══════════════════════════════════════════════════════════════════════════

# Set to True to auto-fill objectives with simulated data (great for learning!)
# Set to False when you want to use your own real experimental data
DEMO_MODE = True

print("✅ Configuration set!")
print(f"\n📋 Your parameters: {list(PARAMETERS.keys())}")
print(f"🎯 Your objectives: {OBJECTIVE_NAMES}")
print(f"📊 Initial samples: {N_INITIAL_SAMPLES}")
print(f"📊 Candidates per round: {N_CANDIDATES}")
print(f"🎮 Demo mode: {'ON (simulated data)' if DEMO_MODE else 'OFF (real data)'}")

---
## Step 3: Generate Initial Design

This creates your first set of experiments using **Latin Hypercube Sampling** - a smart way to spread experiments across your parameter space.

Just run this cell - no editing needed!

In [ ]:
# Set random seed for reproducibility
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Create output folder
output_dir = Path("workshop_results") / CAMPAIGN_NAME
output_dir.mkdir(parents=True, exist_ok=True)

# Prepare parameter bounds for the algorithm
param_names = list(PARAMETERS.keys())
bounds_array = np.array([PARAMETERS[p] for p in param_names])
bounds_tensor = torch.tensor(
    [bounds_array[:, 0].tolist(), bounds_array[:, 1].tolist()],
    dtype=torch.float64
)

# ═══════════════════════════════════════════════════════════════════════════
# SET UP CONSTRAINT FOR INITIAL DESIGN
# ═══════════════════════════════════════════════════════════════════════════
constraint_callable = None
if ConstraintConfig.ENABLE_UREA_CONSTRAINT:
    print(f"🔒 Applying urea constraint to initial design (solubilization_urea={ConstraintConfig.SOLUBILIZATION_UREA} M)")
    from constraints.urea_dilution import urea_constraint_callable as _urea_constraint
    constraint_callable = _urea_constraint

# Generate initial design using Latin Hypercube Sampling with constraint
print("🔄 Generating initial experimental design...")
initial_samples = generate_initial_design(
    n_samples=N_INITIAL_SAMPLES,
    bounds=bounds_tensor,
    seed=RANDOM_SEED,
    n_candidates=100,
    use_maximin=True,
    constraint_callable=constraint_callable
)

# Create DataFrame with the design
df_initial = pd.DataFrame(
    data=initial_samples.numpy(),
    columns=param_names
)

# Add empty columns for objectives (you'll fill these in later!)
for obj_name in OBJECTIVE_NAMES:
    df_initial[obj_name] = np.nan

# Save to Excel
initial_file = output_dir / "Iteration_0_experimental_plan.xlsx"
df_initial.to_excel(initial_file, index=False)

print(f"\n✅ Initial design generated!")
print(f"📁 Saved to: {initial_file}")
print(f"\n📊 Your {N_INITIAL_SAMPLES} initial experiments:")
df_initial[param_names]

In [ ]:
# Visualize the design space with a pairplot
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter

def create_design_space_pairplot(df, bounds, param_names, title="Design Space Coverage", 
                                  n_strata=3, save_path=None):
    """
    Create a pairplot showing design space coverage with stratification grid.
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame containing only the parameter columns
    bounds : array-like
        Array of (min, max) bounds for each parameter
    param_names : list
        List of parameter names
    title : str
        Plot title
    n_strata : int
        Number of strata for grid lines and ticks
    save_path : Path or str
        Where to save the plot
    """
    # Create the pairplot
    pairplot = sns.pairplot(
        df, 
        plot_kws={'edgecolor': 'k', 'linewidth': 0.5, 's': 15, 'alpha': 0.7},
        diag_kws={'color': 'steelblue', 'edgecolor': 'k', 'alpha': 0.7},
        height=1.5, 
        aspect=1
    )
    
    # Compute tick positions based on stratification
    def get_strata_ticks(bounds_i, n_strata):
        """Get tick positions aligned with strata boundaries."""
        return [bounds_i[0] + (k / n_strata) * (bounds_i[1] - bounds_i[0]) 
                for k in range(n_strata + 1)]
    
    # Formatter for 1 decimal place
    formatter = FormatStrFormatter('%.1f')
    
    # Add stratification grid lines and ticks to each subplot
    for i, ax_row in enumerate(pairplot.axes):
        for j, ax in enumerate(ax_row):
            # Get tick positions for this axis
            xticks = get_strata_ticks(bounds[j], n_strata)
            yticks = get_strata_ticks(bounds[i], n_strata)
            
            if i == j:
                # Diagonal: set xlim and ticks
                ax.set_xlim(bounds[j])
                ax.set_xticks(xticks)
                ax.xaxis.set_major_formatter(formatter)
                ax.tick_params(axis='x', rotation=45)
            else:
                # Off-diagonal: set limits, ticks, and add grid lines
                ax.set_xlim(bounds[j])
                ax.set_ylim(bounds[i])
                ax.set_xticks(xticks)
                ax.set_yticks(yticks)
                ax.xaxis.set_major_formatter(formatter)
                ax.yaxis.set_major_formatter(formatter)
                ax.tick_params(axis='x', rotation=45)
                ax.tick_params(axis='y', rotation=45)
                
                # Add dotted stratification lines
                for k in range(1, n_strata):
                    # Vertical lines
                    ax.vlines(
                        x=bounds[j][0] + (k / n_strata) * (bounds[j][1] - bounds[j][0]),
                        ymin=bounds[i][0], ymax=bounds[i][1],
                        colors='grey', linestyles='dotted', linewidth=0.5, alpha=0.5
                    )
                    # Horizontal lines
                    ax.hlines(
                        y=bounds[i][0] + (k / n_strata) * (bounds[i][1] - bounds[i][0]),
                        xmin=bounds[j][0], xmax=bounds[j][1],
                        colors='grey', linestyles='dotted', linewidth=0.5, alpha=0.5
                    )
    
    plt.suptitle(title, y=1.02, fontsize=12)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"📁 Plot saved to: {save_path}")
    
    plt.show()

# Create the pairplot
# Number of strata equals number of samples (LHS property: one sample per stratum)
n_strata = N_INITIAL_SAMPLES

create_design_space_pairplot(
    df=df_initial[param_names],
    bounds=bounds_array,
    param_names=param_names,
    title=f"Initial Design Space Coverage (n={N_INITIAL_SAMPLES})",
    n_strata=3,
    save_path=output_dir / "Iteration_0_design_space_pairplot.png"
)

---
## Step 4: Enter Your Experimental Results

### 🧪 Time to do experiments!

**If DEMO_MODE = True:**
Just run the cell below - it will automatically fill in simulated results!

**If DEMO_MODE = False:**
1. Open the Excel file: `workshop_results/{CAMPAIGN_NAME}/Iteration_0_experimental_plan.xlsx`
2. Perform your experiments according to the parameters
3. Fill in your measured results in the objective columns
4. Save the file
5. Run the cell below to load your results

**Tip:** You can also manually edit the file path below if you saved it somewhere else.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# AUTO-DETECT RESULTS FILE (no need to edit!)
# ═══════════════════════════════════════════════════════════════════════════

# Find the latest iteration that has an experimental plan
latest_iteration = 0
while (output_dir / f"Iteration_{latest_iteration + 1}_experimental_plan.xlsx").exists():
    latest_iteration += 1

# Use the latest iteration's file
results_file = output_dir / f"Iteration_{latest_iteration}_experimental_plan.xlsx"

print(f"📍 Loading results from: Iteration_{latest_iteration}_experimental_plan.xlsx")

# ═══════════════════════════════════════════════════════════════════════════
# Helper function for demo mode
# ═══════════════════════════════════════════════════════════════════════════

def fill_synthetic_results(df: pd.DataFrame, param_names: list, objective_names: list) -> pd.DataFrame:
    """
    Fill objective columns with synthetic data for demo/practice.
    
    This simulates experimental results so you can practice the workflow
    without doing real experiments.
    """
    X = df[param_names].to_numpy()
    
    # Use the synthetic functions from demo_workflow.py
    if len(objective_names) >= 1:
        df[objective_names[0]] = synthetic_objective_1(X)
    if len(objective_names) >= 2:
        df[objective_names[1]] = synthetic_objective_2(X)
    if len(objective_names) > 2:
        # For additional objectives, use a simple combination
        for i in range(2, len(objective_names)):
            df[objective_names[i]] = 0.5 * (synthetic_objective_1(X) + synthetic_objective_2(X))
    
    return df

# ═══════════════════════════════════════════════════════════════════════════
# Load or fill results
# ═══════════════════════════════════════════════════════════════════════════

# Load the experimental plan
df_results = pd.read_excel(results_file)

if DEMO_MODE:
    # Auto-fill with synthetic data
    print("🎮 DEMO MODE: Auto-filling results with simulated data...")
    df_results = fill_synthetic_results(df_results, param_names, OBJECTIVE_NAMES)
    
    # Save the filled results back to Excel
    df_results.to_excel(results_file, index=False)
    print(f"   ✅ Simulated results saved to: {results_file}")
    print("\n📊 Summary of simulated results:")
    print(df_results[OBJECTIVE_NAMES].describe())
else:
    # Check if results are filled in manually
    missing = df_results[OBJECTIVE_NAMES].isna().sum().sum()
    if missing > 0:
        print(f"⚠️ Warning: {missing} objective values are missing (NaN)")
        print("Please fill in your experimental results in the Excel file.")
        print(f"\n📝 Open this file and fill in the objective columns:")
        print(f"   {results_file}")
        print("\nCurrent data:")
        display(df_results)
    else:
        print("✅ All results loaded successfully!")
        print("\n📊 Summary of your results:")
        print(df_results[OBJECTIVE_NAMES].describe())

---
## Step 5: Train the Model

Now we'll train a **Gaussian Process (GP)** model on your data. This model learns the relationship between your parameters and outcomes.

The model will be used to:
1. Predict outcomes for new parameter combinations
2. Quantify uncertainty in predictions
3. Suggest the most informative next experiments

In [ ]:
# Prepare data for training
X_raw = df_results[param_names].to_numpy()
y_raw = df_results[OBJECTIVE_NAMES].to_numpy()

# Normalize and standardize
train_x_normalized, train_y_standardized, scalers = prepare_data(
    X_raw, y_raw, bounds_array
)

# Train a model for each objective
models = []
losses_list = []

print("🔄 Training GP models...")
print(f"   Noise level: {NOISE_LEVEL}")
print(f"   Training iterations: {N_TRAINING_ITERATIONS}")
print()

for i, obj_name in enumerate(OBJECTIVE_NAMES):
    print(f"   Training model {i+1}/{len(OBJECTIVE_NAMES)}: {obj_name}")
    
    train_y_single = train_y_standardized[:, i]
    
    model, likelihood, losses = fit_gp_model(
        train_x=train_x_normalized,
        train_y=train_y_single,
        model_class=GPModel,
        noise=NOISE_LEVEL,
        num_train_iters=N_TRAINING_ITERATIONS,
        lr=0.01
    )
    
    models.append(model)
    losses_list.append(losses)
    print(f"      Final loss: {losses[-1]:.4f}")

print("\n✅ Models trained successfully!")

In [ ]:
# Determine current iteration for file naming
current_iter = 0
while (output_dir / f"Iteration_{current_iter + 1}_experimental_plan.xlsx").exists():
    current_iter += 1

# Plot training loss
fig, axes = plt.subplots(1, len(OBJECTIVE_NAMES), figsize=(10, 3))
if len(OBJECTIVE_NAMES) == 1:
    axes = [axes]

for i, (obj_name, losses) in enumerate(zip(OBJECTIVE_NAMES, losses_list)):
    axes[i].plot(losses)
    axes[i].set_title(f'Training Loss: {obj_name}')
    axes[i].set_xlabel('Iteration')
    axes[i].set_ylabel('Loss')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
loss_plot_path = output_dir / f"Iteration_{current_iter}_training_loss.png"
plt.savefig(loss_plot_path, dpi=150)
plt.show()
print(f"📁 Plot saved to: {loss_plot_path}")

---
## Step 6: Get BO Suggestions

Now the exciting part! The Bayesian optimization will suggest the **most informative** next experiments based on:
- What the model has learned so far
- Where there's high uncertainty
- Where promising results might be found

In [ ]:
# Create multi-output model
multi_model = ModelListGP(*models)

# Set up the acquisition function
# Reference point: values below this are considered "bad"
reference_point = torch.tensor([0.0] * len(OBJECTIVE_NAMES), dtype=torch.float64)

# Create sampler for Monte Carlo integration
sampler = SobolQMCNormalSampler(sample_shape=torch.Size([256]))

# Create acquisition function (qNEHVI - for multi-objective optimization)
acq_function = create_qnehvi_acquisition(
    model=multi_model,
    reference_point=reference_point,
    X_baseline=train_x_normalized,
    sampler=sampler,
    n_objectives=len(OBJECTIVE_NAMES)
)

# Normalized bounds for optimization
normalized_bounds = get_normalized_bounds(len(param_names))

# ═══════════════════════════════════════════════════════════════════════════
# SET UP NONLINEAR CONSTRAINTS (urea dilution physical constraint)
# ═══════════════════════════════════════════════════════════════════════════
nonlinear_inequality_constraints = None
if ConstraintConfig.ENABLE_UREA_CONSTRAINT:
    print(f"🔒 Urea constraint enabled (solubilization_urea={ConstraintConfig.SOLUBILIZATION_UREA} M)")
    # Pass the original bounds so constraint can denormalize internally
    nonlinear_inequality_constraints = [get_urea_constraint_callable(bounds=bounds_tensor)]

print("🔄 Optimizing acquisition function...")
print("   This may take a minute...")

# Optimize to find best next experiments
candidates_normalized = optimize_qnehvi(
    acq_function=acq_function,
    bounds=normalized_bounds,
    batch_size=N_CANDIDATES,
    mc_samples=256,
    num_restarts=20,
    raw_samples=100,
    sequential=True,
    nonlinear_inequality_constraints=nonlinear_inequality_constraints
)

# Convert back to original parameter space
candidates_original = unnormalize(candidates_normalized, bounds_tensor)

# ═══════════════════════════════════════════════════════════════════════════
# VERIFY CONSTRAINT SATISFACTION
# ═══════════════════════════════════════════════════════════════════════════
if ConstraintConfig.ENABLE_UREA_CONSTRAINT:
    print("\n🔍 Verifying constraint satisfaction...")
    for i, candidate in enumerate(candidates_original):
        final_urea = candidate[ConstraintConfig.FINAL_UREA_IDX].item()
        dilution_factor = candidate[ConstraintConfig.DILUTION_FACTOR_IDX].item()
        constraint_value = final_urea * dilution_factor - ConstraintConfig.SOLUBILIZATION_UREA
        status = "✅" if constraint_value > 0 else "⚠️"
        print(f"   Candidate {i+1}: final_urea={final_urea:.2f}, dilution={dilution_factor:.1f}, "
              f"constraint={constraint_value:.2f} {status}")

# Create DataFrame for new experiments
df_suggestions = pd.DataFrame(
    data=candidates_original.numpy(),
    columns=param_names
)

# Add empty columns for objectives
for obj_name in OBJECTIVE_NAMES:
    df_suggestions[obj_name] = np.nan

# ═══════════════════════════════════════════════════════════════════════════
# AUTO-DETECT NEXT ITERATION NUMBER
# ═══════════════════════════════════════════════════════════════════════════

# Find the next available iteration number
next_iteration = 0
while (output_dir / f"Iteration_{next_iteration}_experimental_plan.xlsx").exists():
    next_iteration += 1

# Save suggestions
suggestions_file = output_dir / f"Iteration_{next_iteration}_experimental_plan.xlsx"
df_suggestions.to_excel(suggestions_file, index=False)

print(f"\n✅ BO suggestions generated for Iteration {next_iteration}!")
print(f"📁 Saved to: {suggestions_file}")
print(f"\n🎯 Suggested next {N_CANDIDATES} experiments:")
df_suggestions[param_names]

In [ ]:
# Compare new suggestions with previous experiments
print("📊 Comparison: Previous experiments vs New suggestions")
print("\nPrevious experiments (Iteration 0):")
print(df_results[param_names].describe().loc[['mean', 'min', 'max']])

print("\nNew suggestions (Iteration 1):")
print(df_suggestions[param_names].describe().loc[['mean', 'min', 'max']])

---
## Step 7: Continue the Campaign

### Just run this cell - iteration is detected automatically!

**What this cell does:**
1. Detects the current iteration based on existing files
2. If `DEMO_MODE = True`: Auto-fills simulated results
3. If `DEMO_MODE = False`: Checks for your manual results
4. Combines all data and shows best results so far

**After running:**
- Go back to **Step 5** to retrain
- Then **Step 6** for new suggestions
- Come back here and repeat!

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# AUTO-DETECT CURRENT ITERATION (no need to edit!)
# ═══════════════════════════════════════════════════════════════════════════

# Find the highest iteration number that has an experimental plan
iteration = 0
while (output_dir / f"Iteration_{iteration + 1}_experimental_plan.xlsx").exists():
    iteration += 1

print(f"📍 Detected current iteration: {iteration}")
print(f"   (This will process: Iteration_{iteration}_experimental_plan.xlsx)")

# ═══════════════════════════════════════════════════════════════════════════

# Load the experimental plan for this iteration
current_plan_file = output_dir / f"Iteration_{iteration}_experimental_plan.xlsx"

if not current_plan_file.exists():
    print(f"⚠️ No experimental plan found for iteration {iteration}")
    print("   Make sure you've run Step 6 to generate suggestions first!")
else:
    df_current = pd.read_excel(current_plan_file)
    
    if DEMO_MODE:
        # Auto-fill with synthetic data
        print(f"\n🎮 DEMO MODE: Auto-filling Iteration {iteration} results...")
        df_current = fill_synthetic_results(df_current, param_names, OBJECTIVE_NAMES)
        df_current.to_excel(current_plan_file, index=False)
        print(f"   ✅ Simulated results saved!")
        missing = 0
    else:
        # Check for missing results
        missing = df_current[OBJECTIVE_NAMES].isna().sum().sum()
    
    if missing > 0:
        print(f"\n⚠️ Warning: {missing} objective values are missing")
        print(f"📝 Please fill in your results in:")
        print(f"   {current_plan_file}")
        print("\n   Then run this cell again!")
    else:
        # Combine with all previous data
        all_data = []
        for i in range(iteration + 1):
            file = output_dir / f"Iteration_{i}_experimental_plan.xlsx"
            if file.exists():
                df_i = pd.read_excel(file)
                df_i['Iteration'] = i
                all_data.append(df_i)
        
        df_combined = pd.concat(all_data, ignore_index=True)
        
        # Save combined data
        combined_file = output_dir / f"combined_data_iteration_{iteration}.xlsx"
        df_combined.to_excel(combined_file, index=False)
        
        print(f"\n✅ Combined {len(df_combined)} experiments from {iteration + 1} iterations")
        print(f"📁 Saved to: {combined_file}")
        
        # ═══════════════════════════════════════════════════════════════════════════
        # VISUALIZE DESIGN SPACE COVERAGE ACROSS ALL ITERATIONS
        # ═══════════════════════════════════════════════════════════════════════════
        
        def create_stratified_combined_pairplot(df, bounds, param_names, n_strata=3, 
                                                 title="", save_path=None):
            """
            Create a pairplot showing design space coverage color-coded by iteration.
            
            Parameters:
            -----------
            df : DataFrame
                DataFrame with parameter columns and 'Iteration' column
            bounds : array-like
                Array of (min, max) bounds for each parameter
            param_names : list
                List of parameter names (excludes 'Iteration')
            n_strata : int
                Number of strata for grid lines
            title : str
                Plot title
            save_path : Path or str
                Where to save the plot
            """
            # Drop 'Unnamed: 0' if present
            if 'Unnamed: 0' in df.columns:
                df = df.drop(columns=['Unnamed: 0'])
            
            # Get unique iterations and assign colors
            unique_iterations = sorted(df['Iteration'].unique())
            palette = sns.color_palette("deep", len(unique_iterations))
            iteration_palette = {it: color for it, color in zip(unique_iterations, palette)}
            
            # Create pairplot with hue by iteration
            pairplot = sns.pairplot(
                df, 
                hue='Iteration',
                palette=iteration_palette,
                vars=param_names,
                plot_kws={'edgecolor': 'k', 'linewidth': 0.5, 's': 15, 'alpha': 0.7},
                diag_kws={'edgecolor': 'k', 'alpha': 0.7},
                height=1.5, 
                aspect=1
            )
            
            # Compute tick positions based on stratification
            def get_strata_ticks(bounds_i, n_strata):
                return [bounds_i[0] + (k / n_strata) * (bounds_i[1] - bounds_i[0]) 
                        for k in range(n_strata + 1)]
            
            formatter = FormatStrFormatter('%.1f')
            
            # Add stratification grid lines and ticks
            for i, ax_row in enumerate(pairplot.axes):
                for j, ax in enumerate(ax_row):
                    xticks = get_strata_ticks(bounds[j], n_strata)
                    yticks = get_strata_ticks(bounds[i], n_strata)
                    
                    if i == j:
                        ax.set_xlim(bounds[j])
                        ax.set_xticks(xticks)
                        ax.xaxis.set_major_formatter(formatter)
                        ax.tick_params(axis='x', rotation=45)
                    else:
                        ax.set_xlim(bounds[j])
                        ax.set_ylim(bounds[i])
                        ax.set_xticks(xticks)
                        ax.set_yticks(yticks)
                        ax.xaxis.set_major_formatter(formatter)
                        ax.yaxis.set_major_formatter(formatter)
                        ax.tick_params(axis='x', rotation=45)
                        ax.tick_params(axis='y', rotation=45)
                        
                        # Add stratification lines
                        for k in range(1, n_strata):
                            ax.vlines(
                                x=bounds[j][0] + (k / n_strata) * (bounds[j][1] - bounds[j][0]),
                                ymin=bounds[i][0], ymax=bounds[i][1],
                                colors='grey', linestyles='dotted', linewidth=0.5, alpha=0.5
                            )
                            ax.hlines(
                                y=bounds[i][0] + (k / n_strata) * (bounds[i][1] - bounds[i][0]),
                                xmin=bounds[j][0], xmax=bounds[j][1],
                                colors='grey', linestyles='dotted', linewidth=0.5, alpha=0.5
                            )
            
            plt.suptitle(title, y=1.02, fontsize=12)
            
            if save_path:
                plt.savefig(save_path, dpi=150, bbox_inches='tight')
                print(f"📁 Pairplot saved to: {save_path}")
            
            plt.show()
        
        # Create the combined pairplot
        print("\n📊 Design Space Coverage Across All Iterations:")
        create_stratified_combined_pairplot(
            df=df_combined,
            bounds=bounds_array,
            param_names=param_names,
            n_strata=3,
            title=f"Design Space Coverage (Iterations 0-{iteration}, n={len(df_combined)})",
            save_path=output_dir / f"combined_pairplot_iteration_{iteration}.png"
        )
        
        # Show best results
        print("\n" + "="*60)
        print("🏆 BEST RESULTS SO FAR")
        print("="*60)
        for obj in OBJECTIVE_NAMES:
            best_idx = df_combined[obj].idxmax()
            best_val = df_combined.loc[best_idx, obj]
            best_iter = df_combined.loc[best_idx, 'Iteration']
            best_params = df_combined.loc[best_idx, param_names].to_dict()
            print(f"\n{obj}: {best_val:.3f} (from Iteration {best_iter})")
            for p, v in best_params.items():
                print(f"   {p}: {v:.2f}")
        
        # Update df_results for the next training step
        df_results = df_combined.copy()
        
        print("\n" + "="*60)
        print("➡️ NEXT STEPS")
        print("="*60)
        print("1. Go to Step 5 to retrain the model")
        print("2. Go to Step 6 to get new suggestions")
        print("3. Come back here (Step 7) to continue the loop")
        print("\nThe iteration number will auto-increment! 🎉")

---
## Quick Reference: Full Workflow

### 🎯 NO NEED TO TRACK ITERATION NUMBERS MANUALLY!

The notebook automatically detects which iteration you're on based on existing files.

```
┌─────────────────────────────────────────────────────────────┐
│  SETUP (Step 2)                                             │
│  • Edit parameters, objectives, and settings                │
│  • Set DEMO_MODE = True for practice                        │
│  • Set DEMO_MODE = False for real experiments               │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  ITERATION 0: Initial Design                                │
│  • Step 3: Generate initial design                          │
│  • Step 4: Load/fill results                                │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  REPEAT FOR EACH ITERATION:                                 │
│                                                             │
│  • Step 5: Train model                                      │
│  • Step 6: Get BO suggestions (auto-saves as next iteration)│
│  • Step 7: Load/fill results & combine data                 │
│                                                             │
│  Then go back to Step 5 → repeat!                           │
└─────────────────────────────────────────────────────────────┘
```

### 🚀 Demo Mode Quick Start:
1. Set `DEMO_MODE = True` in Step 2
2. Run Steps 3, 4, 5, 6, 7
3. Repeat Steps 5, 6, 7 as many times as you want
4. No need to change any iteration numbers!

### 📁 Files Created:
- `Iteration_0_experimental_plan.xlsx` - Initial design
- `Iteration_1_experimental_plan.xlsx` - First BO suggestions
- `Iteration_2_experimental_plan.xlsx` - Second BO suggestions
- `combined_data_iteration_X.xlsx` - All data up to iteration X

---
## Tips & Troubleshooting

### Common Issues:

**"Model not learning well"**
- Try increasing `NOISE_LEVEL` if your measurements are noisy
- Try increasing `N_TRAINING_ITERATIONS`

**"Excel file not found"**
- Check the file path in the code
- Make sure you saved the Excel file after editing
